# Laboratori 01: Nga modeli fizik te eksperimenti numerik

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si shndërrohet një ligj fizik në një simulim që mund të verifikohet, të matet dhe të riprodhohet?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- dallojë sistemin real, modelin, algoritmin dhe kodin;
- shënojë supozimet, parametrat, njësitë, hyrjet dhe daljet;
- diskretizojë një ekuacion diferencial me metodën e Eulerit;
- verifikojë kodin me një zgjidhje analitike dhe me studim konvergjence;
- nxjerrë me dorë koston $T(N)$ dhe ta kontrollojë me kohëmatje;
- diskutojë qëndrueshmërinë dhe kufijtë e modelit;
- dallojë verifikimin nga vleftësimi dhe të ruajë riprodhueshmërinë.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Kapitulli i parë nuk synon t'ju mësojë vetëm një integrues. Ai vendos disiplinën e punës që do të ndiqet gjatë gjithë lëndës. Para çdo qelize kodi duhet të mund të thoni cilit nivel i përket ajo: parametrave, modelit, algoritmit, eksperimentit apo paraqitjes grafike. Nëse këto nivele përzihen, një ndryshim i vogël në model kërkon rishkrim të gjithë programit dhe gabimet bëhen të vështira për t'u lokalizuar.

Shembulli i rënies është zgjedhur sepse ka një zgjidhje analitike, jo sepse është fizikisht më i pasuri. Një rast i thjeshtë, ku përgjigjja njihet, shërben si provë për kodin. Vetëm pasi kjo provë të kalojë ka kuptim të shtojmë rezistencë kuadratike, zhurmë ose të dhëna eksperimentale. Ky rend quhet zhvillim progresiv i modelit.


## 1. Katër nivele që nuk duhen ngatërruar

Marrim si sistem real një trup që bie në ajër. Për ta studiuar duhet të bëjmë katër zgjedhje të dallueshme:

1. **Sistemi real:** trupi, ajri, fusha e rëndesës dhe kushtet e matjes.
2. **Modeli:** trup pikësor, $g$ konstant dhe rezistencë lineare $-cv$.
3. **Algoritmi:** diskretizimi i kohës dhe rregulli i Eulerit.
4. **Kodi:** zbatimi i rregullit me vargje, cikle dhe funksione.

Kodi mund ta zbatojë saktë një model të papërshtatshëm. Prandaj pyetja “a punon programi?” ndahet në dy pyetje: “a zgjidh kodi ekuacionet e zgjedhura?” dhe “a përshkruajnë këto ekuacione eksperimentin real?”.


## 2. Modeli dhe kontrolli dimensional

Zgjedhim drejtimin pozitiv për poshtë. Ligji i dytë i Njutonit jep

$$m\frac{dv}{dt}=mg-cv,\qquad \frac{dy}{dt}=v.$$

Njësitë janë $[m]=\mathrm{kg}$, $[v]=\mathrm{m\,s^{-1}}$, $[g]=\mathrm{m\,s^{-2}}$ dhe $[c]=\mathrm{kg\,s^{-1}}$. Raporti

$$\tau=\frac{m}{c}$$

ka njësi kohe dhe mat sa shpejt sistemi harron shpejtësinë fillestare. Shpejtësia kufitare është

$$v_\infty=\frac{mg}{c}.$$

Zgjidhja analitike e shpejtësisë është

$$v(t)=v_\infty+(v_0-v_\infty)e^{-t/\tau}.$$

Kjo zgjidhje do të përdoret për **verifikim**; nuk është e dhënë eksperimentale dhe, prandaj, nuk kryen vleftësim.


### 2.1 Llogaritja e parë me dorë

Për $m=1\,\mathrm{kg}$, $c=0.5\,\mathrm{kg\,s^{-1}}$, $g=9.81\,\mathrm{m\,s^{-2}}$ dhe $v_0=0$:

$$\tau=\frac{1}{0.5}=2\,\mathrm{s},\qquad
v_\infty=\frac{1\cdot9.81}{0.5}=19.62\,\mathrm{m\,s^{-1}}.$$

Në $t=2\,\mathrm{s}=\tau$,

$$v(\tau)=v_\infty(1-e^{-1})\approx0.632v_\infty.$$

Para kodit, parashikoni: a mund ta kalojë shpejtësia numerike $v_\infty$ kur nis nga $v_0=0$ dhe përdoret një hap i arsyeshëm?


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


## 3. Parametrat si të dhëna, jo si numra të fshehur

Parametrat mblidhen në një strukturë. Kjo e bën të qartë se çfarë ndryshon nga një eksperiment te tjetri dhe ruan njësitë në komente. `frozen=True` pengon ndryshimin aksidental gjatë simulimit.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Parametra:
    m: float = 1.0       # masa [kg]
    c: float = 0.5       # koeficienti i rezistencës [kg/s]
    g: float = 9.81      # nxitimi i rëndesës [m/s^2]
    y0: float = 0.0      # pozicioni fillestar [m]
    v0: float = 0.0      # shpejtësia fillestare [m/s]
    T: float = 8.0       # koha përfundimtare [s]

p = Parametra()
print(p)
print('tau =', p.m/p.c, 's; v_inf =', p.m*p.g/p.c, 'm/s')


## 4. Nga derivati te metoda e Eulerit

Për $h=\Delta t$, zhvillimi Taylor jep

$$v(t+h)=v(t)+h\frac{dv}{dt}+\frac{h^2}{2}v''(\xi).$$

Duke hequr termin e rendit $h^2$ dhe duke zëvendësuar ekuacionin diferencial:

$$v_{n+1}=v_n+h\left(g-\frac{c}{m}v_n\right),$$

$$y_{n+1}=y_n+hv_n.$$

Gabimi lokal i një hapi është $O(h^2)$; në një interval të fiksuar kryhen afërsisht $T/h$ hapa, prandaj gabimi global bëhet $O(h)$. Ky argument duhet shkruar para eksperimentit të konvergjencës.


In [ ]:
def renie_euler(p, h):
    N = int(np.ceil(p.T / h))
    h = p.T / N                      # arrijmë saktësisht kohën T
    t = np.linspace(0.0, p.T, N + 1)
    y = np.empty(N + 1)
    v = np.empty(N + 1)
    y[0], v[0] = p.y0, p.v0

    for n in range(N):
        y[n + 1] = y[n] + h*v[n]     # përdor vlerën në fillim të hapit
        v[n + 1] = v[n] + h*(p.g - (p.c/p.m)*v[n])
    return t, y, v

t, y, v = renie_euler(p, h=0.1)
print('N =', len(t)-1, 'v(T) =', v[-1], 'm/s')


In [ ]:
def v_analitike(t, p):
    tau = p.m/p.c
    v_inf = p.m*p.g/p.c
    return v_inf + (p.v0-v_inf)*np.exp(-t/tau)

v_ref = v_analitike(t, p)
gabimi_max = np.max(np.abs(v-v_ref))
print(f'gabimi maksimal = {gabimi_max:.6e} m/s')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, v_ref, 'k-', label='zgjidhja analitike')
ax.plot(t, v, 'o', ms=3, markevery=5, label='Euler')
ax.set(xlabel='koha $t$ [s]', ylabel='shpejtësia $v$ [m/s]')
ax.grid(alpha=0.3); ax.legend(); plt.show()


## 5. Studimi i konvergjencës

Nëse $E(h)\approx Ch^p$, atëherë

$$\frac{E(h)}{E(h/2)}\approx2^p,qquad
p\approx\log_2\frac{E(h)}{E(h/2)}.$$

Për Euler-in presim $p\approx1$. Vetëm krahasimi i dy kurbave nuk mjafton; llogarisim gabimin dhe rendin e vëzhguar.


In [ ]:
hapat = np.array([0.4, 0.2, 0.1, 0.05, 0.025])
gabimet = []
for h in hapat:
    t_h, _, v_h = renie_euler(p, h)
    gabimet.append(np.max(np.abs(v_h-v_analitike(t_h, p))))
gabimet = np.array(gabimet)
rendet = np.log2(gabimet[:-1]/gabimet[1:])

for i, h in enumerate(hapat):
    shtese = '' if i == 0 else f', p≈{rendet[i-1]:.3f}'
    print(f'h={h:7.4f}, E={gabimet[i]:.6e}{shtese}')


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(hapat, gabimet, 'o-', label='gabimi i matur')
ax.loglog(hapat, gabimet[-1]*(hapat/hapat[-1]), '--', label='$O(h)$')
ax.set(xlabel='hapi $h$ [s]', ylabel='gabimi maksimal [m/s]')
ax.grid(True, which='both', alpha=0.3); ax.legend(); plt.show()


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 6.1 Kompleksiteti i Eulerit me dorë

Në çdo hap kryhen një numër konstant mbledhjesh dhe shumëzimesh. Për $N=T/h$ hapa,

$$T(N)=aN+b\in\Theta(N),$$

ndërsa ruhen tri vargje me $N+1$ elemente, prandaj kujtesa është $\Theta(N)$. Nëse na duhet vetëm gjendja përfundimtare, mund të ruajmë dy skalarë dhe kujtesa bëhet $\Theta(1)$.

Përgjysmimi i $h$ dyfishon afërsisht $N$ dhe kohën, por përgjysmon gabimin global të Eulerit. Ky është një shkëmbim i matshëm midis kostos dhe saktësisë.


In [ ]:
def kohematje(f, *args, perseritje=5):
    kohet = []
    for _ in range(perseritje):
        t0 = perf_counter(); f(*args); kohet.append(perf_counter()-t0)
    return min(kohet)

N_lista = np.array([200, 400, 800, 1600, 3200, 6400])
kohet = []
for N in N_lista:
    kohet.append(kohematje(renie_euler, p, p.T/N))
kohet = np.array(kohet)
pjerr = np.polyfit(np.log(N_lista), np.log(kohet), 1)[0]
print(f'pjerrësia e matur p = {pjerr:.3f}; pritja teorike = 1')


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 7.1 Qëndrueshmëria e Eulerit për pjesën homogjene

Për devijimin $w=v-v_\infty$ kemi $w'=-(c/m)w$. Euler-i jep

$$w_{n+1}=\left(1-h\frac{c}{m}\right)w_n=G w_n.$$

Qëndrueshmëria kërkon $|G|<1$, pra

$$0<h<\frac{2m}{c}=2\tau.$$

Ky kusht nuk thotë se çdo hap nën kufi është mjaft i saktë; thotë vetëm se perturbimet nuk rriten pa kufi. Saktësia dhe qëndrueshmëria janë kritere të ndryshme.


In [ ]:
h_lista = [0.5*p.m/p.c, 1.5*p.m/p.c, 2.1*p.m/p.c]
fig, ax = plt.subplots(figsize=(7, 4))
for h in h_lista:
    t_h, _, v_h = renie_euler(p, h)
    G = 1-h*p.c/p.m
    ax.plot(t_h, v_h, 'o-', label=f'h={h:.2f}, G={G:.2f}')
ax.axhline(p.m*p.g/p.c, color='k', ls='--', label='$v_\\infty$')
ax.set(xlabel='t [s]', ylabel='v [m/s]', title='Qëndrueshmëria e Eulerit')
ax.grid(alpha=0.3); ax.legend(); plt.show()


## 8. Shembull i plotë: verifikim, ndjeshmëri dhe vleftësim

Do të ndajmë tri burime mospërputhjeje:

- gabimin e diskretizimit, duke ndryshuar $h$;
- ndjeshmërinë ndaj $c$, duke perturbur parametrin;
- mospërputhjen me “matje” sintetike, që imiton vleftësimin.

Ndjeshmëria qendrore ndaj $c$ është

$$S_c(t)\approx\frac{v(t;c+\Delta c)-v(t;c-\Delta c)}{2\Delta c}.$$


In [ ]:
from dataclasses import replace

dc = 0.01*p.c
p_plus = replace(p, c=p.c+dc)
p_minus = replace(p, c=p.c-dc)
t_s = np.linspace(0, p.T, 200)
S_c = (v_analitike(t_s, p_plus)-v_analitike(t_s, p_minus))/(2*dc)

# Matje sintetike: model + zhurmë e njohur.
t_mat = np.linspace(0, p.T, 25)
v_mat = v_analitike(t_mat, p) + rng.normal(0, 0.15, size=t_mat.size)
mbetjet = v_mat-v_analitike(t_mat, p)
print('RMSE ndaj matjeve sintetike =', np.sqrt(np.mean(mbetjet**2)), 'm/s')


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(t_s, S_c, label=r'$S_c$')
ax[0].set(xlabel='t [s]', ylabel='$S_c$ [(m/s)/(kg/s)]', title='Ndjeshmëria ndaj $c$')
ax[1].plot(t_s, v_analitike(t_s, p), label='modeli')
ax[1].errorbar(t_mat, v_mat, yerr=0.15, fmt='o', ms=4, label='matje sintetike')
ax[1].set(xlabel='t [s]', ylabel='v [m/s]', title='Ideja e vleftësimit')
for a in ax: a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.show()


## 9. Punishtja — ndrysho vetëm një supozim

Zgjidhni një rast:

1. ndryshoni $c$ dhe shpjegoni ndryshimin e $\tau$ dhe $v_\infty$;
2. nisni me $v_0>v_\infty$ dhe parashikoni drejtimin e evolucionit;
3. përdorni një hap pranë kufirit $2m/c$;
4. zëvendësoni rezistencën lineare me $-cv|v|$ dhe identifikoni çfarë nuk mund të verifikohet më me formulën aktuale.

Kodi vijues paraqet dinamikisht trajektoren kur `DINAMIK=True`. Në ekzekutim automatik përdoret figura përfundimtare.


In [ ]:
def punishte(p, h=0.1, DINAMIK=False):
    t, y, v = renie_euler(p, h)
    fig, ax = plt.subplots(figsize=(7, 4))
    if DINAMIK:
        for k in range(2, len(t)+1, max(1, len(t)//30)):
            ax.clear(); ax.plot(t[:k], v[:k], color='tab:blue')
            ax.axhline(p.m*p.g/p.c, color='k', ls='--')
            ax.set(xlabel='t [s]', ylabel='v [m/s]'); ax.grid(alpha=0.3)
            plt.pause(0.03)
    else:
        ax.plot(t, v, label='Euler')
        ax.plot(t, v_analitike(t, p), '--', label='analitike')
        ax.set(xlabel='t [s]', ylabel='v [m/s]'); ax.grid(alpha=0.3); ax.legend()
    plt.show()

punishte(p, h=0.1, DINAMIK=False)


### Kontroll para kodimit: shkallët pa përmasa

Me $u=v/v_\infty$ dhe $s=t/\tau$, ekuacioni bëhet

$$\frac{du}{ds}=1-u.$$

Modeli pa përmasa nuk përmban $m$, $c$ ose $g$ veçmas. Kjo tregon se të gjitha sistemet me rezistencë lineare kanë të njëjtën kurbë $u(s)$ pasi përdoren shkallët e duhura. Nëse dy simulime me parametra të ndryshëm nuk bien mbi të njëjtën kurbë pa përmasa, kodi ose shkallëzimi duhet kontrolluar.


In [ ]:
# Kontrolli i kolapsit pa përmasa për dy bashkësi parametrash.
from dataclasses import replace
p2=replace(p,m=2.0,c=0.25,g=9.81,T=20.0)
for pp in [p,p2]:
    t0,_,v0=renie_euler(pp,0.02)
    s=t0/(pp.m/pp.c); u=v0/(pp.m*pp.g/pp.c)
    print('maksimumi |u-(1-exp(-s))| =',np.max(np.abs(u-(1-np.exp(-s)))))


### Testet numerike si pohime të ekzekutueshme

Një kontroll që mbetet vetëm në tekst mund të harrohet. Pohimet `assert` e bëjnë kriterin pjesë të programit. Ato duhet të shprehin veti fizike ose matematikore të qarta: kushtin fillestar, monotoninë kur $v_0<v_\infty$, arritjen e kohës përfundimtare dhe zvogëlimin e gabimit kur hapi përgjysmohet. Testi nuk provon gjithçka; ai mbron një veti të deklaruar.


In [ ]:
t_a,_,v_a=renie_euler(p,.1); t_b,_,v_b=renie_euler(p,.05)
Ea=np.max(np.abs(v_a-v_analitike(t_a,p)))
Eb=np.max(np.abs(v_b-v_analitike(t_b,p)))
assert np.isclose(t_a[0],0) and np.isclose(t_a[-1],p.T)
assert np.isclose(v_a[0],p.v0)
assert np.all(np.diff(v_a)>=-1e-12)
assert Eb<Ea
print('Të katër testet kaluan; raporti E(h)/E(h/2)=',Ea/Eb)


### Çfarë duhet të shkruhet pas një figure

Interpretimi duhet të ndajë tri shtresa. Së pari jepet vëzhgimi sasior: për shembull, gabimi përgjysmohet afërsisht kur përgjysmohet $h$. Së dyti lidhet vëzhgimi me teorinë: kjo është sjellja e pritshme e një metode me rend global një. Së treti jepet kufiri: kjo provë verifikon diskretizimin për modelin linear, por nuk vërteton se forca lineare e rezistencës përshkruan një trup real në çdo shpejtësi.


In [ ]:
# Tabelë e shkurtër që mund të kopjohet në raport.
print(' h [s]      N       E_max [m/s]    E(h)/E(h/2)')
for i,h in enumerate(hapat):
    ratio=np.nan if i==len(hapat)-1 else gabimet[i]/gabimet[i+1]
    print(f'{h:7.4f} {int(np.ceil(p.T/h)):7d} {gabimet[i]:14.6e} {ratio:12.4f}')


### Pyetje për vetëkontroll

1. Cili ndryshim kërkon model të ri: zvogëlimi i $h$ apo zëvendësimi i $-cv$ me $-cv|v|$?
2. Pse një studim konvergjence nuk është vleftësim?
3. Nëse ruhet vetëm $v(T)$, si ndryshon kujtesa pa ndryshuar kompleksitetin kohor?
4. Cili test do të dështonte nëse shenja e forcës së rezistencës shkruhej gabim?


# Pjesa II — GNU Octave

## 10. Zbatimi thelbësor

Kodi Octave ruan të njëjtën ndarje midis parametrave, algoritmit dhe kontrollit. Vargjet indeksohen nga 1; kjo ndryshon sintaksën, jo metodën.

```octave
m=1.0; c=0.5; g=9.81; y0=0.0; v0=0.0; T=8.0; h=0.1;
N=ceil(T/h); h=T/N; t=linspace(0,T,N+1);
y=zeros(1,N+1); v=zeros(1,N+1); y(1)=y0; v(1)=v0;

for n=1:N
    y(n+1)=y(n)+h*v(n);
    v(n+1)=v(n)+h*(g-(c/m)*v(n));
end

tau=m/c; vinf=m*g/c;
vref=vinf+(v0-vinf)*exp(-t/tau);
fprintf('gabimi maksimal = %.6e m/s\n', max(abs(v-vref)));
plot(t,v,'o-',t,vref,'k--'); grid on;
xlabel('t [s]'); ylabel('v [m/s]'); legend('Euler','analitike');
```


## 11. Konvergjenca dhe stabiliteti në Octave

```octave
hs=[0.4,0.2,0.1,0.05,0.025]; E=zeros(size(hs));
for j=1:length(hs)
    h=hs(j); N=ceil(T/h); h=T/N; t=linspace(0,T,N+1);
    v=zeros(1,N+1); v(1)=v0;
    for n=1:N
        v(n+1)=v(n)+h*(g-(c/m)*v(n));
    end
    vref=vinf+(v0-vinf)*exp(-t/tau);
    E(j)=max(abs(v-vref));
end
rend=log2(E(1:end-1)./E(2:end));
disp([hs(:),E(:)]); disp('rendet:'); disp(rend);
```

Krahasoni rendet me $1$ dhe provoni një hap $h>2m/c$. Mos e ngatërroni shpërthimin numerik me një dukuri fizike.


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = 'import numpy as np\n\ndef euler(m,c,g,v0,T,h):\n    N=int(np.ceil(T/h)); h=T/N\n    t=np.linspace(0,T,N+1); v=np.empty(N+1); v[0]=v0\n    for n in range(N): v[n+1]=v[n]+h*(g-(c/m)*v[n])\n    return t,v\n\nm,c,g,v0,T=1.0,0.5,9.81,0.0,8.0\nfor h in [0.2,0.1,0.05]:\n    t,v=euler(m,c,g,v0,T,h)\n    ref=m*g/c+(v0-m*g/c)*np.exp(-t/(m/c))\n    print(h,np.max(np.abs(v-ref)))\n'
script_path = script_dir / 'laboratori_01_modeli_dhe_eksperimenti.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_01_modeli_dhe_eksperimenti.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Dallova sistemin real, modelin, algoritmin dhe kodin.
- [ ] Kontrollova njësitë dhe nxora $\tau$ e $v_\infty$.
- [ ] Mata rendin e konvergjencës dhe kompleksitetin.
- [ ] Dallova verifikimin nga vleftësimi.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
